# MOMENT trên PTB-XL: forecasting (fcst) + Membership Inference (TV4)

Pipeline: tải PTB-XL → chia member / non-member → fine-tune MOMENT-1-large cho **forecasting** (lookback 512, horizon 96, full fine-tune, loss MSE) → score Loss = `-MSE` theo từng bản ghi → TPR @ 0,1% và 1% FPR + bootstrap CI → ghi `results/foundation/fcst/loss_<mode>.json` lên Drive. Dự đoán của model được lưu lại (`*_preds_ep{n}.pt`) để làm tấn công Temporal (DFT + trend) sau này.

Chỉ có attack **Loss** vì forecasting không có softmax (không có Confidence/Entropy).

**Trước khi chạy**
1. Runtime → Change runtime type → GPU (A100 hoặc L4).
2. Upload lên Google Drive, trong `MyDrive/Final_Project/`: `moment/` và `dataset/` giống notebook cls.
3. Nếu session bị ngắt: Run all lại, notebook tự chạy tiếp từ checkpoint trong `Final_Project/ckpt/foundation_fcst/`.
4. Split dùng chung với notebook cls (cùng `ptbxl_{LEVEL}_s{SEED}.json`), nên member/non-member giống hệt nhau giữa hai task.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # must be set before torch touches CUDA
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Final_Project'
MOMENT_SRC = f'{DRIVE_ROOT}/moment'          # folder containing setup.py
DATASET_DIR = f'{DRIVE_ROOT}/dataset'        # shared split script + splits/
PTBXL_ZIP = f'{DATASET_DIR}/archive (3).zip'
RESULTS_DIR = f'{DRIVE_ROOT}/results/foundation/fcst'

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copy source to local disk (faster than installing from Drive), then install
!rm -rf /content/moment && cp -r {MOMENT_SRC} /content/moment
!pip install -q /content/moment wfdb scikit-learn pandas tqdm

## 1. Cấu hình
`LEVEL = 'record'` chia ngẫu nhiên theo bản ghi (stratified).
Split dùng chung của cả nhóm do `dataset/make_split.py` tạo (train = member, test = non-member, val = tập nhỏ lấy từ non-member). Nếu `dataset/splits/` chưa có file tương ứng, notebook tự chạy script để tạo.

In [ ]:
SEED = 42
LEVEL = 'record'            # record-level split
MODEL_NAME = 'AutonLab/MOMENT-1-large'
MODES = ['full']

LOOKBACK = 512             # MOMENT input length
HORIZON = 96               # forecast steps; the dataset window is LOOKBACK + HORIZON

EPOCHS = 10
EVAL_AT = [1, 2, 5, 10]     # epochs at which MIA is evaluated (RQ2: dependence on #steps)
BATCH = 16                  # lower to 8 if CUDA OOM persists (gradient checkpointing is already on)
MAX_LR = {'full': 1e-4}

N_VAL = 500                 # must match the split file (make_split.py --n-val)
N_BOOT = 1000
CKPT_EVERY = 1              # save a resumable checkpoint to Drive every N epochs (~4.5GB)
CKPT_DIR = f'{DRIVE_ROOT}/ckpt/foundation_fcst'

PTBXL_ROOT = '/content/ptbxl_raw'
SPLIT_PATH = f'{DATASET_DIR}/splits/ptbxl_{LEVEL}_s{SEED}.json'
CACHE_DIR = f'/content/cache/fcst_{LEVEL}_s{SEED}'

## 2. Dữ liệu PTB-XL
Giải nén `archive (3).zip` từ Drive vào disk local của Colab (đọc nhanh hơn Drive).

In [ ]:
import os, glob
if not glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True):
    os.makedirs(PTBXL_ROOT, exist_ok=True)
    assert os.path.exists(PTBXL_ZIP), f'missing {PTBXL_ZIP}; {DATASET_DIR} has {os.listdir(DATASET_DIR)}'
    !unzip -q -o "{PTBXL_ZIP}" -d {PTBXL_ROOT}
found = glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True)
assert found, f'no ptbxl_database.csv after unzip; {PTBXL_ROOT} has {os.listdir(PTBXL_ROOT)}'
PTBXL_DIR = os.path.dirname(found[0])
print(PTBXL_DIR, os.listdir(PTBXL_DIR))

In [ ]:
import json, random, subprocess, sys
import numpy as np, torch, pandas as pd
from momentfm.data.ptbxl_classification_dataset import PTBXL_dataset

def control_randomness(seed=SEED):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
control_randomness()

# shared split from dataset/make_split.py (generated here if missing)
if not os.path.exists(SPLIT_PATH):
    subprocess.run([sys.executable, f'{DATASET_DIR}/make_split.py', '--ptbxl-dir', PTBXL_DIR,
                    '--level', LEVEL, '--seed', str(SEED), '--n-val', str(N_VAL)], check=True)
split = json.load(open(SPLIT_PATH))
meta = json.load(open(SPLIT_PATH.replace('.json', '.meta.json')))
print({k: len(v) for k, v in split.items()}, 'sha256', meta['sha256'][:12])
if len(split['test']) < 10000:
    print('WARNING: fewer than 10k non-members; TPR@0.1%FPR rests on', int(0.001 * len(split['test'])), 'samples')

class Config:
    base_path = PTBXL_DIR
    cache_dir = CACHE_DIR
    path_to_pids = SPLIT_PATH
    load_cache = True
    fs = 100
    output_type = 'single'    # lowercase: modify_df compares against 'single'
    seq_len = LOOKBACK + HORIZON   # record is resampled to this length, then split into input / target
os.makedirs(CACHE_DIR, exist_ok=True)

member_ds = PTBXL_dataset(Config, phase='train')
val_ds    = PTBXL_dataset(Config, phase='val')
nonmem_ds = PTBXL_dataset(Config, phase='test')
print(len(member_ds), len(val_ds), len(nonmem_ds), member_ds.timeseries.shape)

## 3. Train + tấn công
- `full`: mở khoá patch embedding + encoder + head.
- Mỗi bản ghi (12 lead) được nội suy thành `LOOKBACK + HORIZON` điểm: `LOOKBACK` điểm đầu là input, `HORIZON` điểm cuối là target.
- Score: **cao hơn = nhiều khả năng là member**. Loss: `-MSE` trung bình trên 12 lead × horizon.

In [ ]:
import time
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve, auc as sk_auc
from momentfm import MOMENTPipeline

dev = 'cuda'
bf16 = torch.cuda.get_device_capability()[0] >= 8
amp = lambda: torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=bf16)

def build_model(mode):
    model = MOMENTPipeline.from_pretrained(MODEL_NAME, model_kwargs={
        'task_name': 'forecasting', 'forecast_horizon': HORIZON,
        'freeze_encoder': False, 'freeze_embedder': False, 'freeze_head': False,
        'enable_gradient_checkpointing': True})
    model.init()
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f'[{mode}] trainable {n_tr/1e6:.2f}M / {n_all/1e6:.2f}M')
    return model.to(dev)

def split_xy(x):
    x = x.to(dev).float()
    return x[..., :LOOKBACK], x[..., LOOKBACK:]

def forward_fcst(model, x_in):
    mask = torch.ones(x_in.shape[0], x_in.shape[-1], device=dev)
    return model(x_enc=x_in, input_mask=mask).forecast.float()

@torch.no_grad()
def get_preds(model, ds, bs=64):
    model.eval()
    out = []
    for x, _ in DataLoader(ds, batch_size=bs, shuffle=False):
        x_in, _ = split_xy(x)
        with amp():
            out.append(forward_fcst(model, x_in).cpu())
    return torch.cat(out)                       # [N, 12, HORIZON]

def targets_of(ds):
    return torch.as_tensor(ds.timeseries[..., LOOKBACK:]).float()

def per_sample_mse(preds, ds):
    return ((preds - targets_of(ds)) ** 2).mean(dim=(1, 2)).numpy()

def attack_scores(preds, ds):
    return {'loss': -per_sample_mse(preds, ds)}

def tpr_at(fpr, tpr, t):
    m = fpr <= t
    return float(tpr[m].max()) if m.any() else 0.0

def mia_metrics(sm, sn):
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
    return tpr_at(fpr, tpr, 1e-3), tpr_at(fpr, tpr, 1e-2), float(sk_auc(fpr, tpr))

def bootstrap_ci(sm, sn, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    res = np.array([mia_metrics(sm[rng.integers(0, len(sm), len(sm))],
                                sn[rng.integers(0, len(sn), len(sn))])[:2] for _ in range(n)])
    lo, hi = np.percentile(res, [2.5, 97.5], axis=0)
    return [float(lo[0]), float(hi[0])], [float(lo[1]), float(hi[1])]

In [ ]:
def save_atomic(obj, path):
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)   # a disconnect mid-write never corrupts the previous checkpoint
    # Force Drive FUSE to flush to server — prevents silent write loss on large files
    with open(path, 'rb') as _f:
        os.fsync(_f.fileno())
    assert os.path.getsize(path) > 256, f'save_atomic: {path} suspiciously small after write'  

def run_mode(mode):
    os.makedirs(CKPT_DIR, exist_ok=True)
    tag = f'{mode}_{LEVEL}_s{SEED}'
    ckpt_path, done_path = f'{CKPT_DIR}/{tag}.pt', f'{CKPT_DIR}/{tag}.done.json'
    key = f"fcst|{LOOKBACK}|{HORIZON}|{MODEL_NAME}|{LEVEL}|{SEED}|{EPOCHS}|{BATCH}|{MAX_LR[mode]}|{meta['sha256'][:12]}"
    if os.path.exists(done_path):
        d = json.load(open(done_path))
        if d['key'] == key:
            print(f'[{mode}] already finished, skipping'); return d['history']

    control_randomness()
    model = build_model(mode)
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    loader = DataLoader(member_ds, batch_size=BATCH, shuffle=True, drop_last=True)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=MAX_LR[mode] / 25)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=MAX_LR[mode], total_steps=EPOCHS * len(loader))
    mse = torch.nn.MSELoss()
    history, start = [], 1

    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location='cpu')
        if ck['key'] == key:
            model.load_state_dict(ck['model'], strict=False)   # only trainable tensors were saved
            opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
            history, start = ck['history'], ck['epoch'] + 1
            print(f"[{mode}] resumed after epoch {ck['epoch']}")
        else:
            print(f'[{mode}] checkpoint has a different config, ignoring it')
        del ck

    for ep in range(start, EPOCHS + 1):
        model.train()
        losses = []
        for x, yb in tqdm(loader, desc=f'{mode} ep{ep}/{EPOCHS}', leave=False):
            opt.zero_grad(set_to_none=True)
            x_in, y_tgt = split_xy(x)
            with amp():
                pred = forward_fcst(model, x_in)
            loss = mse(pred, y_tgt)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step()
            losses.append(loss.item())

        if ep in EVAL_AT:
            pm, pn = get_preds(model, member_ds), get_preds(model, nonmem_ds)
            save_atomic({'pm': pm, 'pn': pn}, f'{CKPT_DIR}/{tag}_preds_ep{ep}.pt')
            sm, sn = attack_scores(pm, member_ds), attack_scores(pn, nonmem_ds)
            mse_m, mse_n = float(-sm['loss'].mean()), float(-sn['loss'].mean())
            row = {'mode': mode, 'epoch': ep, 'train_loss': float(np.mean(losses)),
                   'train_mse': mse_m, 'test_mse': mse_n, 'gap': mse_n - mse_m}
            for a in sm:
                t01, t1, au = mia_metrics(sm[a], sn[a])
                row.update({f'{a}_tpr@0.1': t01, f'{a}_tpr@1': t1, f'{a}_auc': au})
            history.append(row)
            print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

        if ep % CKPT_EVERY == 0 or ep == EPOCHS or ep in EVAL_AT:
            save_atomic({'key': key, 'epoch': ep, 'history': history, 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                         'model': {k: v.cpu() for k, v in model.state_dict().items() if k in trainable}}, ckpt_path)

    # final epoch = last evaluated epoch; predictions come from disk so this also works right after a resume
    final_ep = max(e for e in EVAL_AT if e <= EPOCHS)
    row = [r for r in history if r['epoch'] == final_ep][-1]
    pr = torch.load(f'{CKPT_DIR}/{tag}_preds_ep{final_ep}.pt')
    sm, sn = attack_scores(pr['pm'], member_ds), attack_scores(pr['pn'], nonmem_ds)
    # write JSON for the last evaluated epoch, schema = results/_sample.json
    os.makedirs(RESULTS_DIR, exist_ok=True)
    for a in sm:
        c01, c1 = bootstrap_ci(sm[a], sn[a])
        out = {
            'model': 'foundation', 'task': 'fcst', 'attack': a,
            'owner_tv': 4, 'level': LEVEL, 'dataset': 'ptbxl', 'seed': SEED,
            'n_member': int(len(sm[a])), 'n_nonmember': int(len(sn[a])),
            'scores_member': np.round(sm[a], 6).tolist(), 'scores_nonmember': np.round(sn[a], 6).tolist(),
            'tpr_at_0.1fpr': row[f'{a}_tpr@0.1'], 'tpr_at_1fpr': row[f'{a}_tpr@1'], 'auc': row[f'{a}_auc'],
            'bootstrap_ci': {'n_resamples': N_BOOT, 'confidence': 0.95, 'tpr_at_0.1fpr': c01, 'tpr_at_1fpr': c1},
            'utility': {'train_mse': row['train_mse'], 'test_mse': row['test_mse'], 'gap': row['gap']},
            'n_shadow': None,
            'extra': {'mode': mode, 'epochs': EPOCHS, 'eval_epoch': row['epoch'], 'max_lr': MAX_LR[mode],
                      'batch': BATCH, 'backbone': MODEL_NAME, 'lookback': LOOKBACK, 'horizon': HORIZON},
        }
        fn = f'{RESULTS_DIR}/{a}_{mode}.json'
        json.dump(out, open(fn, 'w'))
        print('saved', fn)

    json.dump({'key': key, 'history': history}, open(done_path, 'w'))
    if os.path.exists(ckpt_path): os.remove(ckpt_path)   # free Drive space; delete the .done.json to retrain
    del model, opt; torch.cuda.empty_cache()
    return history

In [ ]:
all_hist = []
for mode in MODES:
    t0 = time.time()
    all_hist += run_mode(mode)
    print(f'{mode} done in {(time.time() - t0) / 60:.1f} min')

hist_df = pd.DataFrame(all_hist)
hist_df.to_csv(f'{RESULTS_DIR}/history_{LEVEL}.csv', index=False)
hist_df

## 4. Tóm tắt
ROC log–log của full fine-tune (attack Loss = -MSE, epoch cuối).

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 5))
for mode in MODES:
    r = json.load(open(f'{RESULTS_DIR}/loss_{mode}.json'))
    sm, sn = np.array(r['scores_member']), np.array(r['scores_nonmember'])
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
    ax.plot(fpr, tpr, label=f"{mode} (AUC {r['auc']:.3f}, TPR@0.1% {r['tpr_at_0.1fpr']:.3f})")
ax.plot([1e-4, 1], [1e-4, 1], 'k--', lw=0.8)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(1e-4, 1); ax.set_ylim(1e-4, 1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=8); ax.set_title(f'MOMENT PTB-XL fcst h={HORIZON}, {LEVEL}-level, Loss attack')
plt.show()